In [0]:
%pip install psycopg2-binary pandas

In [0]:
# ============================================================
# NOTEBOOK 000 - EXTRAÇÃO (Landing Zone) - VERSÃO PYSPARK NATIVO
# ============================================================

# 1. Credenciais (Pooler IPv4 do Supabase)
DB_HOST     = "aws-1-sa-east-1.pooler.supabase.com"
DB_PORT     = "5432"
DB_NAME     = "postgres"
DB_USER     = "postgres.zbjndadbyodcqjlqvvyo"
DB_PASSWORD = "oNbyCpIL22PFXoL4" # <-- Coloque sua senha!

# 2. Montando a URL JDBC com SSL forçado
jdbc_url = f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require"

connection_properties = {
    "user": DB_USER,
    "password": DB_PASSWORD,
    "driver": "org.postgresql.Driver"
}

# 3. Lista das tabelas do seu projeto
TABELAS = [
    'apolice', 'carro', 'cliente', 'endereco', 'estado', 
    'marca', 'modelo', 'municipio', 'regiao', 'sinistro', 'telefone'
]

CAMINHO_LANDING = '/Volumes/workspace/landing/dados'

print("Iniciando extração nativa via PySpark...")

for tabela in TABELAS:
    try:
        print(f"Extraindo tabela: {tabela}...")
        
        # O Spark lê a tabela sem sobrecarregar a memória do kernel
        df_spark = spark.read.jdbc(url=jdbc_url, table=tabela, properties=connection_properties)
        
        # Salvando na Landing Zone
        (df_spark
            .coalesce(1) # Força a gerar um único arquivo
            .write
            .mode("overwrite")
            .option("header", "true")
            .option("sep", ",")
            .csv(f"{CAMINHO_LANDING}/{tabela}_temp")
        )
        
        # Renomeando o arquivo para ficar limpo (ex: cliente.csv)
        files = dbutils.fs.ls(f"{CAMINHO_LANDING}/{tabela}_temp")
        csv_file = [f.path for f in files if f.name.endswith('.csv')][0]
        dbutils.fs.cp(csv_file, f"{CAMINHO_LANDING}/{tabela}.csv")
        dbutils.fs.rm(f"{CAMINHO_LANDING}/{tabela}_temp", recurse=True)
        
        print(f"  ✓ {tabela}.csv salvo com sucesso!")
        
    except Exception as e:
        print(f"  ❌ Erro ao extrair a tabela {tabela}: {e}")

print("\n✅ Processo finalizado!")

In [0]:
# Caminho do Volume Landing no Databricks
CAMINHO_LANDING = '/Volumes/workspace/landing/dados'

# JDBC URL para PostgreSQL
jdbc_url = f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Propriedades de conexão
connection_properties = {
    "user": DB_USER,
    "password": DB_PASSWORD,
    "driver": "org.postgresql.Driver"
}

# Lista das tabelas a extrair (ajuste conforme seu banco)
TABELAS = [
    'apolice',
    'carro',
    'cliente',
    'endereco',
    'estado',
    'marca',
    'modelo',
    'municipio',
    'regiao',
    'sinistro',
    'telefone'
]

print("Iniciando extração do PostgreSQL...")

for tabela in TABELAS:
    print(f"  Extraindo tabela: {tabela}...")
    
    # Ler tabela com Spark JDBC
    df_spark = spark.read.jdbc(
        url=jdbc_url,
        table=tabela,
        properties=connection_properties
    )
    
    # Salvar como CSV no Volume Landing
    (df_spark
        .coalesce(1)  # Gera um único arquivo CSV
        .write
        .mode("overwrite")
        .option("header", "true")
        .option("sep", ",")
        .csv(f"{CAMINHO_LANDING}/{tabela}_temp")
    )
    
    # Renomear para nome limpo (Databricks gera nome com hash)
    files = dbutils.fs.ls(f"{CAMINHO_LANDING}/{tabela}_temp")
    csv_file = [f.path for f in files if f.name.endswith('.csv')][0]
    dbutils.fs.cp(csv_file, f"{CAMINHO_LANDING}/{tabela}.csv")
    dbutils.fs.rm(f"{CAMINHO_LANDING}/{tabela}_temp", recurse=True)
    
    print(f"    ✓ {tabela}.csv salvo com {df_spark.count()} registros")

print("\n✅ Extração concluída! Todos os CSVs estão no Volume Landing.")

In [0]:
display(dbutils.fs.ls('/Volumes/workspace/landing/dados/'))